
This notebook demonstrates fine-tuning a pre-trained transformer model from Hugging Face for translation tasks with several improvements:

1. Proper train/validation split
2. BLEU score evaluation during training
3. Better preprocessing with language-specific prefixes
4. More efficent training arguments
5. Model saving and loading demonstration

In [ ]:
import os

import torch

import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments

from datasets import load_dataset, DatasetDict

import sacrebleu

print("Libraries imported successfully")

In [ ]:
# Load the WMT14 dataset

print("Loading WMT14 dataset...")

raw_datasets = load_dataset("wmt14", "de-en")



# Show dataset structure

print("Dataset splits:")

for split in raw_datasets:

    print(f"  {split}: {len(raw_datasets[split])} examples")



# For faster training, we'll use a subset of the training data

# but keep the full validation set for evaluation

train_size = min(10000, len(raw_datasets["train"]))  # Use up to 10k training examples

train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(train_size))

val_dataset = raw_datasets["validation"]  # Use full validation set



print(f"\nUsing {len(train_dataset)} training examples and {len(val_dataset)} validation examples")

In [ ]:
# Load tokenizer and model from Hugging Face

model_name = "t5-small"



tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)



print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

print(f"Model config: {model.config}")

In [ ]:
# Preprocessing function with language-specific prefix

max_input_length = 128

max_target_length = 128



def preprocess_function(examples):

    # Add translation prefix as recommended for T5

    inputs = ["translate English to German: " + ex["translation"]["en"] for ex in examples]

    targets = [ex["translation"]["de"] for ex in examples]

    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    

    # Setup the tokenizer for targets

    with tokenizer.as_target_tokenizer():

        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs



# Apply preprocessing

print("Tokenizing training dataset...")

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)



print("Tokenizing validation dataset...")

tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)



print("Tokenized dataset shapes:")

print(f"  Train: {len(tokenized_train)} examples")

print(f"  Validation: {len(tokenized_val)} examples")

In [ ]:
# Function to compute BLEU score

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    if isinstance(preds, tuple):

        preds = preds[0]

    # Replace -100 in the labels as we can't decode them

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    

    # Decode predictions and labels

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    

    # sacrebleu expects list of references for each prediction

    # We have a 1:1 mapping, so we wrap each label in a list

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])

    

    return {"bleu": bleu.score}


In [ ]:
# Define training arguments with improved settings

training_args = Seq2SeqTrainingArguments(

    output_dir="./results",

    evaluation_strategy="epoch",

    learning_rate=3e-4,  # Slightly higher learning rate for T5

    per_device_train_batch_size=16,  # Increased batch size

    per_device_eval_batch_size=16,

    weight_decay=0.01,

    save_total_limit=3,

    num_train_epochs=3,

    predict_with_generate=True,

    fp16=torch.cuda.is_available(),

    logging_dir="./logs",

    logging_steps=50,

    load_best_model_at_end=True,  # Load the best model at the end of training

    metric_for_best_model="bleu",  # Use BLEU to determine best model

    greater_is_better=True,  # Higher BLEU is better

    report_to="none",  # Disable wandb/comet logging for simplicity

)



print("Training arguments defined.")

In [ ]:
# Define data collator (for seq2seq)
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Initialize Trainer
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # Add metrics computation
)

print("Trainer initialized.")

In [ ]:
# Start training

print("Starting training...")
trainer.train()



print("Training completed.")

In [ ]:
# Evaluate on validation set

print("Evaluating on validation set...")

eval_results = trainer.evaluate()

print(f"Validation results: {eval_results}")



# Save the fine-tuned model

print("Saving model...")

trainer.save_model("./fine_tuned_t5_small")

tokenizer.save_pretrained("./fine_tuned_t5_small")



print("Model saved to ./fine_tuned_t5_small")

In [ ]:
# Example inference with the fine-tuned model

from transformers import pipeline



# Load the fine-tuned model

translator = pipeline("translation", model="./fine_tuned_t5_small", tokenizer="./fine_tuned_t5_small", device=0 if torch.cuda.is_available() else -1)



# Test translation

test_sentences = [

    "Hello, how are you?",

    "The weather is nice today.",

    "I love machine learning."

]



print("\nTranslation examples:")

for sentence in test_sentences:

    result = translator(f"translate English to German: {sentence}")[0]['translation_text']

    print(f"EN: {sentence}")

    print(f"DE: {result}")

    print()